# Preprocessing for Evaluation

In order to execute this notebook you need to upload a **finetuned model + tokenizer** (zip file). The notebook generates two **.tsv files** that can be used for further evaluation in **evaluation.ipynb**

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install transformers==4.51.3
    !pip install --no-deps unsloth

In [ ]:
from unsloth import FastLanguageModel
import re
from datasets import load_dataset, concatenate_datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip drive/MyDrive/lora_multi_e3_225.zip -d lora

Archive:  drive/MyDrive/lora_multi_e3_225.zip
   creating: lora/lora/
  inflating: lora/lora/tokenizer.json  
  inflating: lora/lora/adapter_config.json  
  inflating: lora/lora/README.md     
  inflating: lora/lora/special_tokens_map.json  
  inflating: lora/lora/tokenizer_config.json  
  inflating: lora/lora/added_tokens.json  
  inflating: lora/lora/adapter_model.safetensors  
  inflating: lora/lora/vocab.json    
  inflating: lora/lora/merges.txt    


In [ ]:
system_prompt = """
You are an advanced language model tasked with detoxifying texts across multiple languages.
Your goal is to identify and neutralize toxic content while preserving the original meaning and language of the input text.

## Definition of Toxicity

Toxic language includes expressions that are offensive, hateful, abusive, discriminatory, or threatening.
It can be explicit, such as direct insults or slurs, or implicit, such as passive-aggressive comments or coded language.
Toxicity reduces the quality of conversation and harms communities.

## Steps to Success

1. Identify toxic content in the text while considering its context
2. Reformulate the text so it becomes a non-toxic text in the initial language
3. Check if it still contains toxic content
4. Make sure that the semantic content of the non-toxic text remains as similar as possible to the original text
5. Output only the detoxified final text

## Examples of Toxic Input and Non-Toxic Output Text Pairs in English

Toxic: "then all of a sudden i see her , shes now got the big phony tits and everything."
Non-toxic: "All of a sudden i see her, she is all grown up."

Toxic: "i would have lost my shit * years * ago."
Non-toxic: "I would have gone crazy years ago."

Toxic: "How about you fuck off and don't stalk my edits?"
Non-toxic: "How about you leave me alone and don't stalk my edits?"

Toxic: "You dumb motherfucker I am going to sue the shit out of you"
Non-toxic: "I am going to sue everything out of you"

Toxic: "таких тварей только могила исправит."
Non-toxic: "Этих людей ничего не исправит"

Toxic: "Wer die Grünen wählt ist entweder dumm oder ein Hurensohn #Schlafschafe"
Non-toxic: "Wer die Grünen wählt ist entweder uninformiert oder naiv"

## Task

Please detoxify the following toxic text using all the things provided above:
"""

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora",
    max_seq_length = 2048,
    load_in_4bit = True,
)
model.eval()

In [ ]:
def extract_cleaned(decoded_output):
    matches = re.findall(r'<think>\s*</think>\s*\n*\s*(.*?)$', decoded_output.strip(), re.DOTALL)
    cleaned = matches[-1].strip() if matches else ''
    cleaned = re.sub(r'^[\+\-\•\#\*\~\s]+', '', cleaned)
    return cleaned

def template(input):
  return [
    {"role" : "user", "content" : system_prompt + input},
  ]

In [ ]:
dataset = load_dataset("textdetox/multilingual_paradetox")
#dataset['en'] = dataset['en'].map(lambda x:{"lang": 'en'})
#dataset['de'] = dataset['de'].map(lambda x:{"lang": 'de'})
#dataset['ru'] = dataset['ru'].map(lambda x:{"lang": 'ru'})
dataset['es'] = dataset['es'].map(lambda x:{"lang": 'es'})
dataset['uk'] = dataset['uk'].map(lambda x:{"lang": 'uk'})
dataset['zh'] = dataset['zh'].map(lambda x:{"lang": 'zh'})

ds = concatenate_datasets([dataset["zh"], dataset["es"], dataset["uk"]], axis=0)
ds = ds.shuffle(seed = 3407).train_test_split(test_size=0.25, seed=42)["test"]
#ds = concatenate_datasets([
#    dataset["en"].shuffle(seed = 3407).train_test_split(test_size=0.25, seed=42)["test"],
#    dataset["de"].shuffle(seed = 3407).train_test_split(test_size=0.25, seed=42)["test"],
#    dataset["ru"].shuffle(seed = 3407).train_test_split(test_size=0.25, seed=42)["test"]
#    ], axis=0)
#ds = dataset['en']

In [ ]:
eval_data = ds # test_dataset

In [ ]:
def create_tsv(model, tokenizer, eval_data):
  print(len(eval_data))
  reference_df = eval_data.to_pandas()
  reference_df.to_csv("reference.tsv", sep="\t", index=False)

  evaluation_df = reference_df.copy()
  evaluation_df['neutral_sentence'] = ""
  print(len(evaluation_df))

  input = evaluation_df["toxic_sentence"].copy()
  input = input.map(lambda x: template(x))
  input = input.to_list()
  detoxified = []
  len_input = len(input)
  for i in range(0, len_input, 1):
    print(f"{i}/{len_input}")
    text = tokenizer.apply_chat_template(
        input[i],
        tokenize = False,
        add_generation_prompt = True,
        enable_thinking = False,
    )
    outputs = model.generate(
        **tokenizer(text, return_tensors = "pt").to("cuda"),
        max_new_tokens = 256,
        temperature = 0.7, top_p = 0.8, top_k = 20,
    )
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    output_cleaned = extract_cleaned(output_text)
    print(output_cleaned)
    detoxified.append(output_cleaned)
  evaluation_df["neutral_sentence"] = detoxified
  evaluation_df.to_csv('lora.tsv', sep="\t", index=False, encoding="utf-8", errors='backslashreplace')

  return evaluation_df, reference_df

In [ ]:
evaluation_df, reference_df = create_tsv(model, tokenizer, eval_data) #<-----------

300
300
0/300
En ESPAÑA tenemos un Gobierno PSOE-UP, asi que simplemente estamos DISFRUTANDO LO QUE SE HA VOTADO.
1/300
Дякую всім, хто тримав кулаки, але співбесіда нє удалась і мене проігнорували.
2/300
粿籹凌迟
3/300
Expulsado #Guillermo Herrera
4/300
YoSoyAnonymousUE Basta de represión absurda por parte del estado
5/300
我不是男同，我也觉得恶心，那我是不是和你一样？
6/300
搞清因果关系，主要问题还是在幕刃这边，打破她们的外包装现出原形就好了，比如把她们的头发P光光，衣服换成男装，口红脂粉全擦掉，再看她们如何卖钱
7/300
Їй випав Овен, мій папа овен
8/300
Так нечестно на очах
9/300
А чи не гад ти, Дмитро?
10/300
Para el podemitariado y los tontos del noreste español, Cristina Pardo también es facha.
11/300
不会真有人想和小仙女结婚吧
12/300
Marhuenda, ese que dice que es catalán solo cuando pretende dañar a todos los catalanes, llama...
13/300
Algunos pierden la calma por llegar a Paguitalandia.
14/300
La justificación de JAJAJAJ mejor no te defiendas 🤣🤣
15/300
你那个性别要不要先改一下??女拳不都喜欢冒充男的说话吗??
16/300
Dios mi madre es muy poco inteligente en serio
17/300
成天带妹的能不能死啊
18/300
Son los mejores. Paguita, a